In [14]:
import pandas as pd
import numpy as np
import os

In [ ]:
RAW_DIR = "data/raw"
CLEAN_DIR = "data/clean"
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(CLEAN_DIR, exist_ok=True)

original_path = os.path.join(RAW_DIR, "Insurance_claims_data.csv")
if not os.path.exists(original_path):
    import shutil
    if os.path.exists("Insurance claims data.csv"):
        shutil.copy("Insurance claims data.csv", original_path)
        print("📦 已将原始数据复制到 data/raw/ 文件夹中。")

df = pd.read_csv(original_path)
print(f"✅ 已读取原始数据：{df.shape[0]} 行，{df.shape[1]} 列")

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

print(f"🔢 数值型变量数量: {len(numeric_cols)}")
print(f"🔤 分类型变量数量: {len(categorical_cols)}")



✅ 已读取原始数据：58592 行，41 列
🔢 数值型变量数量: 13
🔤 分类型变量数量: 28


In [3]:
# ========== 4️⃣ 数值型变量清洗 ==========
for col in numeric_cols:
    mean_val = df[col].mean()
    df[col].fillna(mean_val, inplace=True)
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    df.loc[(df[col] < lower) | (df[col] > upper), col] = mean_val

print("✅ 数值型变量缺失值与异常值处理完成")

# ========== 5️⃣ 分类变量清洗 ==========
for col in categorical_cols:
    mode_val = df[col].mode()[0] if not df[col].mode().empty else "Unknown"
    df[col].fillna(mode_val, inplace=True)

print("✅ 分类变量缺失值处理完成")

# ========== 6️⃣ 去重 + 客户 ID 模拟 ==========
df = df.drop_duplicates(subset=['policy_id']).reset_index(drop=True)
policy_count = len(df)
target_customer_count = int(policy_count / 1.5)  # 平均 1.5 保单 / 客户

np.random.seed(42)
df['customer_id'] = np.random.randint(1, target_customer_count + 1, size=policy_count)

unique_customers = df['customer_id'].nunique()
print(f"🧍‍♂️ 模拟生成客户数量: {unique_customers}（约 {policy_count/unique_customers:.2f} 保单/客户）")

# ========== 7️⃣ 保存清洗结果（覆盖原文件） ==========
clean_path = os.path.join(CLEAN_DIR, "Insurance_claims_data_cleaned.csv")
df.to_csv(clean_path, index=False)

C:\Users\86133\AppData\Local\Temp\ipykernel_24436\2309722179.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(mean_val, inplace=True)
C:\Users\86133\AppData\Local\Temp\ipykernel_24436\2309722179.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when

✅ 数值型变量缺失值与异常值处理完成
✅ 分类变量缺失值处理完成
🧍‍♂️ 模拟生成客户数量: 30349（约 1.93 保单/客户）


In [4]:
print("\n🎯 数据清洗完成并已覆盖保存！")
print(f"📂 清洗后文件路径: {clean_path}")
print("示例:")
print(df[['policy_id', 'customer_id', 'customer_age']].head(10))


🎯 数据清洗完成并已覆盖保存！
📂 清洗后文件路径: data/clean\Insurance_claims_data_cleaned.csv
示例:
   policy_id  customer_id  customer_age
0  POL045360        15796          41.0
1  POL016745          861          35.0
2  POL007194        38159          44.0
3  POL018146        11285          44.0
4  POL049011         6266          56.0
5  POL053680        16851          36.0
6  POL053943        37195          38.0
7  POL002857        21963          56.0
8  POL028225        16024          55.0
9  POL047631         1686          45.0


In [17]:
df1 = pd.read_csv("data/clean/Insurance_claims_data_cleaned.csv")
df1['region_code'].value_counts()

region_code
C8     13654
C2      7342
C5      6979
C3      6101
C14     3660
C13     3423
C10     3155
C9      2734
C7      2167
C12     1589
C1      1468
C11     1212
C19      952
C6       890
C15      771
C4       665
C17      492
C16      401
C21      379
C18      242
C22      207
C20      109
Name: count, dtype: int64

In [12]:
import pandas as pd
import numpy as np

csv = "data/clean/Insurance_claims_data_cleaned.csv"
df  = pd.read_csv(csv)

# 1) 快速修正：非0都当作 1（也可用阈值 >=0.5）
df['claim_status'] = (df['claim_status'] != 0).astype(int)

# 2) 保险：只允许 {0,1}
assert set(df['claim_status'].unique()).issubset({0,1})

# 3) 覆盖保存（或另存新文件）
df.to_csv(csv, index=False)

print(df['claim_status'].value_counts(dropna=False))


claim_status
0    54844
1     3748
Name: count, dtype: int64


In [13]:
import pyodbc

conn = pyodbc.connect(
    "Driver={ODBC Driver 18 for SQL Server};"
    "Server=tcp:dongjing-sql-server.database.windows.net,1433;"
    "Database=insurance_db;Uid=sqladmin;Pwd=YourStrongPassword123!;"
    "Encrypt=yes;TrustServerCertificate=no;"
)
cur = conn.cursor()
cur.fast_executemany = True

rows = list(df[['policy_id', 'claim_status']].itertuples(index=False, name=None))
cur.executemany("""
    UPDATE dbo.Fact_Policy
    SET claim_status = ?
    WHERE policy_id = ?
""", [(int(cs), pid) for pid, cs in rows])  # 注意参数顺序
conn.commit()
cur.close(); conn.close()


In [18]:
import pandas as pd

region_map = pd.DataFrame({
    "region_code": ["C8","C2","C5","C3","C14","C13","C10","C9","C7","C12",
                    "C1","C11","C19","C6","C15","C4","C17","C16","C21","C18","C22","C20"],
    "state_name": ["California","Texas","Florida","New York","Illinois","Pennsylvania",
                   "Ohio","Georgia","North Carolina","Michigan","Washington","Arizona",
                   "Massachusetts","Virginia","New Jersey","Colorado","Tennessee",
                   "Indiana","Missouri","Minnesota","Wisconsin","Oregon"],
    "state_abbr": ["CA","TX","FL","NY","IL","PA","OH","GA","NC","MI",
                   "WA","AZ","MA","VA","NJ","CO","TN","IN","MO","MN","WI","OR"]
})

region_map.to_csv("data/region_mapping.csv", index=False)
print("✅ region_mapping.csv 已保存！")


✅ region_mapping.csv 已保存！
